# Data filler

Since the generation ENTSO-E data have many NaN values, the purpose of this notebook is to investigate the NaN values of the processed datasets and eventually fill them, in order to allow the most complete and significant data analysis in the following stage.

The data filling process is didvided by zone:
1. [Italy - National Data](#1-italy---national-data)  
    1.1. [Load dataset and define analysis sample](#11-load-dataset-and-define-analysis-sample)  
    1.2. [Prelminary missing-data investigation](#12-preliminary-missing-data-investigation)
    1.3. [Prelminary missing-data investigation](#12-preliminary-missing-data-investigation)
    1.2. [Prelminary missing-data investigation](#12-preliminary-missing-data-investigation)


## 1. Italy - National Data

In [2]:
import pandas as pd 
import warnings
from pathlib import Path

# filter User and Filter Warnings for privacy matters
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

### 1.1. Load dataset and define analysis sample

In [3]:
# load dataset
dataset_path = Path('../dataset/processed')
df = pd.read_csv(dataset_path / 'IT.csv')

# fix datetime timezones 
df['datetime_UTC'] = pd.to_datetime(df['datetime_UTC'], utc=True)
df['datetime_local'] = pd.to_datetime(df['datetime_local'], utc=True)
df['datetime_local'] = df['datetime_local'].dt.tz_convert('Europe/Rome')

In [4]:
# filter dataset by period of time
start = '2016-01-01 00:00'
end = '2026-06-30 23:00'

start_date = pd.Timestamp(
    f'{start}',
    tz='Europe/Rome'
)

end_date = pd.Timestamp(
    f'{end}',
    tz='Europe/Rome'
)

df = df[
    df['datetime_local'].between(start_date, end_date)
].copy()

In [5]:
df

,datetime_UTC,datetime_local,local_hour,PUN,National Load,Biomass,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,...,Hydro Pumped Storage,Hydro Run-of-river and poundage,Hydro Water Reservoir,Other,Solar,Waste,Wind Onshore,Wind Offshore,Energy storage,Other renewable
8736,2015-12-31 23:00:00+00:00,2016-01-01 00:00:00+01:00,0,50.41211,23927.233,220.00,241.00,5588.00,1686.00,542.00,...,0.0,1259.00,287.00,9112.00,0.00,45.0,203.00,NaN,NaN,NaN
8737,2016-01-01 00:00:00+00:00,2016-01-01 01:00:00+01:00,1,44.76435,22517.279,216.00,241.00,5034.00,1719.00,519.00,...,0.0,1163.00,81.00,9324.00,1.00,46.0,218.00,NaN,NaN,NaN
8738,2016-01-01 01:00:00+00:00,2016-01-01 02:00:00+01:00,2,40.01613,21354.678,217.00,241.00,4486.00,1728.00,530.00,...,0.0,1089.00,20.00,8739.00,0.00,44.0,267.00,NaN,NaN,NaN
8739,2016-01-01 02:00:00+00:00,2016-01-01 03:00:00+01:00,3,37.92076,20219.874,216.00,241.00,4275.00,1727.00,467.00,...,0.0,1010.00,20.00,8195.00,0.00,39.0,227.00,NaN,NaN,NaN
8740,2016-01-01 03:00:00+00:00,2016-01-01 04:00:00+01:00,4,35.77287,19229.221,215.00,241.00,3926.00,1658.00,426.00,...,0.0,988.00,20.00,7876.00,0.00,44.0,212.00,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100746,2026-06-30 17:00:00+00:00,2026-06-30 19:00:00+02:00,19,204.10622,48437.300,712.50,641.50,26968.25,394.75,155.75,...,1464.5,4189.75,1167.25,1709.00,1755.75,NaN,1780.75,1.00,1148.333333,NaN
100747,2026-06-30 18:00:00+00:00,2026-06-30 20:00:00+02:00,20,198.16000,47383.500,714.00,661.75,27442.75,396.50,156.00,...,1257.5,4343.25,1209.75,1767.00,252.00,NaN,2076.50,5.75,1273.750000,NaN
100748,2026-06-30 19:00:00+00:00,2026-06-30 21:00:00+02:00,21,182.03250,45951.100,716.50,676.50,27084.00,392.25,156.00,...,998.0,4252.50,982.50,1793.50,0.50,NaN,2587.75,5.50,830.333333,NaN
100749,2026-06-30 20:00:00+00:00,2026-06-30 22:00:00+02:00,22,173.30306,43204.200,710.25,647.50,26959.75,365.75,158.25,...,832.0,3931.75,618.00,1792.25,0.00,NaN,2068.75,1.25,241.750000,NaN


### 1.2. Preliminary missing-data investigation

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 92015 entries, 8736 to 100750
Data columns (total 21 columns):
 #   Column                           Non-Null Count  Dtype                      
---  ------                           --------------  -----                      
 0   datetime_UTC                     92015 non-null  datetime64[us, UTC]        
 1   datetime_local                   92015 non-null  datetime64[us, Europe/Rome]
 2   local_hour                       92015 non-null  int64                      
 3   PUN                              92015 non-null  float64                    
 4   National Load                    92015 non-null  float64                    
 5   Biomass                          92015 non-null  float64                    
 6   Fossil Coal-derived gas          91417 non-null  float64                    
 7   Fossil Gas                       91870 non-null  float64                    
 8   Fossil Hard coal                 91515 non-null  float64                 

In [7]:
# get columns and counts of missing values
missing = df.isna().sum()
missing = missing[missing > 0]

print('\n--- MISSING VALUES ---')
print(missing)


--- MISSING VALUES ---
Fossil Coal-derived gas      598
Fossil Gas                   145
Fossil Hard coal             500
Fossil Oil                   755
Geothermal                  8147
Hydro Pumped Storage        7298
Hydro Water Reservoir         24
Other                        288
Waste                       2351
Wind Onshore                   8
Wind Offshore              56111
Energy storage             82821
Other renewable            87506
dtype: int64


In [ ]:
# get completeness rate by year from generation columns
generation_cols = [
    'Biomass',
    'Fossil Coal-derived gas',
    'Fossil Gas',
    'Fossil Hard coal',
    'Fossil Oil',
    'Geothermal',
    'Hydro Pumped Storage',
    'Hydro Run-of-river and poundage',
    'Hydro Water Reservoir',
    'Other',
    'Solar',
    'Waste',
    'Wind Onshore',
    'Wind Offshore',
    'Energy storage',
    'Other renewable'
]

df['year'] = df['datetime_local'].dt.year

completeness = (
    df.groupby('year')[generation_cols]
    .agg(lambda x: x.notna().mean() * 100)
    .round(2)
    .reset_index()
)

completeness

,year,Biomass,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,Geothermal,Hydro Pumped Storage,Hydro Run-of-river and poundage,Hydro Water Reservoir,Other,Solar,Waste,Wind Onshore,Wind Offshore,Energy storage,Other renewable
0,2016,100.0,100.00,98.62,100.00,100.00,100.00,99.76,100.0,100.00,100.00,100.0,100.00,100.00,0.00,0.00,0.00
1,2017,100.0,100.00,99.73,98.90,100.00,100.00,99.89,100.0,100.00,100.00,100.0,100.00,100.00,0.00,0.00,0.00
2,2018,100.0,99.73,100.00,99.73,100.00,100.00,99.99,100.0,100.00,98.08,100.0,100.00,100.00,0.00,0.00,0.00
3,2019,100.0,93.70,100.00,100.00,99.73,100.00,97.15,100.0,100.00,98.63,100.0,100.00,100.00,0.00,0.00,0.00
4,2020,100.0,100.00,100.00,100.00,100.00,100.00,95.34,100.0,100.00,100.00,100.0,100.00,100.00,0.00,0.00,0.00
5,2021,100.0,100.00,100.00,100.00,100.00,100.00,95.51,100.0,99.73,100.00,100.0,100.00,100.00,0.00,0.00,0.00
6,2022,100.0,100.00,100.00,100.00,100.00,100.00,93.79,100.0,100.00,100.00,100.0,100.00,100.00,60.01,0.00,0.00
7,2023,100.0,100.00,100.00,100.00,100.00,100.00,94.98,100.0,100.00,100.00,100.0,100.00,100.00,100.00,0.00,0.00
8,2024,100.0,100.00,100.00,100.00,91.79,91.25,95.09,100.0,100.00,100.00,100.0,100.00,100.00,100.00,0.00,0.00
9,2025,100.0,100.00,100.00,100.00,99.99,25.64,75.18,100.0,100.00,100.00,100.0,99.18,99.91,100.00,82.39,51.47


In [ ]:
def missing_intervals(df, col, time_col='datetime_UTC'):
    """
    Identifies consecutive intervals of missing values.
    """
    # get column with missing values
    missing = df[col].isna()

    # assign ID to each consecutive block
    block_id = missing.ne(missing.shift()).cumsum()

    # keep only rows with missing values
    missing_rows = df.loc[missing, [time_col]].copy()
    missing_rows['block_id'] = (
        block_id.loc[missing].to_numpy()
    )

    # summarise each missing interval
    intervals = (
        missing_rows
        .groupby('block_id')
        .agg(
            start=(time_col, 'min'),
            end=(time_col, 'max'),
            n_observations=(time_col, 'size')
        )
        .sort_values('start')
    )

    return intervals

detailed_cols = [
    'Fossil Coal-derived gas', 
    'Fossil Gas', 
    'Fossil Hard coal', 
    'Fossil Oil', 
    'Geothermal', 
    'Hydro Water Reservoir', 
    'Other', 
    'Waste', 
    'Wind Onshore', 
    'Wind Offshore', 
    'Wind Offshore'
]

# execute function
for col in detailed_cols:
    print(f'\n--- {col} ---')
    print(missing_intervals(df, col).to_string(index=False))


--- Fossil Coal-derived gas ---
                    start                       end  n_observations
2018-12-30 23:00:00+00:00 2018-12-31 22:00:00+00:00              24
2019-01-01 23:00:00+00:00 2019-01-24 22:00:00+00:00             552
2026-04-27 15:00:00+00:00 2026-04-28 12:00:00+00:00              22

--- Fossil Gas ---
                    start                       end  n_observations
2016-10-26 22:00:00+00:00 2016-10-31 22:00:00+00:00             121
2017-06-25 22:00:00+00:00 2017-06-26 21:00:00+00:00              24

--- Fossil Hard coal ---
                    start                       end  n_observations
2017-05-30 22:00:00+00:00 2017-06-01 21:00:00+00:00              48
2017-06-20 22:00:00+00:00 2017-06-22 21:00:00+00:00              48
2018-12-30 23:00:00+00:00 2018-12-31 22:00:00+00:00              24
2026-02-10 14:00:00+00:00 2026-02-11 07:00:00+00:00              18
2026-02-16 09:00:00+00:00 2026-02-16 22:00:00+00:00              14
2026-02-17 00:00:00+00:00 2026-02-17 

### 1.3. Documented missing-data treatments
The following issues were identified during the investigation of missing values in the Italian national generation dataset. The proposed treatments aim to improve data completeness while avoiding unsupported imputations.

1.	**Incomplete generation data in 2015**  
Hirth et al. (2018) documented substantial completeness issues in ENTSOE generation data for Italy. Using data retrieved in 2017, the authors reported that Italian generation observations were absent for 2015 and appeared in 2016, resulting in an approximately 50% share of missing observations across the two years considered.
The current ENTSOE download contains a different completeness pattern, indicating that historical data availability has changed since the original assessment. Nevertheless, several generation technologies still display incomplete coverage during the second half of 2015.  
For the purposes of the main analysis, 2015 is excluded to avoid introducing a period with uneven historical coverage. The analysis sample therefore runs from 1 January 2016 to 30 June 2026. The exclusion of 2015 is a methodological choice rather than a claim that every series is unusable during that year.  
2.	**Wind offshore**  
The ENTSOE Wind Offshore series contains missing observations before the appearance of the first operational offshore wind farm in Italy, Beleolico, inaugurated in Taranto on 21 April 2022 (Renexia, 2022).  
Missing observations before the technology became operational can be interpreted as structural zeros, provided that the absence of offshore generation during that period is supported by the available evidence. Missing observations during the commissioning period, before the first reported ENTSOE value, are retained unless their actual production status can be established.  

In [11]:
# fill wind offshore nan
df_test = df.copy()

# define the cutoff date in Italian local time
cutoff = pd.Timestamp(
    '2022-04-21 00:00:00',
    tz='Europe/Rome'
)

# select only missing Wind Offshore values before the cutoff
mask = (
    (df_test['datetime_local'] < cutoff)
    & (df_test['Wind Offshore'].isna())
)

# fill only the selected values with zero
df_test.loc[mask, 'Wind Offshore'] = 0

print('Values filled with zero:', mask.sum())
print(
    'Remaining Wind Offshore NaN:',
    df_test['Wind Offshore'].isna().sum()
)

Values filled with zero: 55247
Remaining Wind Offshore NaN: 864


In [12]:
# check
df_test.loc[
    df_test['datetime_local'].between(
        pd.Timestamp('2022-04-15', tz='Europe/Rome'),
        pd.Timestamp('2022-06-01', tz='Europe/Rome')
    ),
    ['datetime_local', 'Wind Offshore']
]

,datetime_local,Wind Offshore
63839,2022-04-15 00:00:00+02:00,0.0
63840,2022-04-15 01:00:00+02:00,0.0
63841,2022-04-15 02:00:00+02:00,0.0
63842,2022-04-15 03:00:00+02:00,0.0
63843,2022-04-15 04:00:00+02:00,0.0
...,...,...
64963,2022-05-31 20:00:00+02:00,0.0
64964,2022-05-31 21:00:00+02:00,0.0
64965,2022-05-31 22:00:00+02:00,0.0
64966,2022-05-31 23:00:00+02:00,0.0


3.	**Energy storage and Hydro Pumped Storage**  
Energy storage and Hydro Pumped Storage have incomplete historical coverage. They are excluded from the primary definition of renewable generation because they represent storagerelated technologies rather than primary renewable energy sources.  
Their treatment in the totalgeneration denominator is defined separately. In particular, hydro pumped storage generation is not equivalent to pumping consumption and may contribute to observed electricity generation. The selected definition of total generation must therefore be stated explicitly.  
4.	**Other renewable only appear in 2024-2025**  
The Other renewable category has insufficient historical coverage and its detailed composition has not yet been established for the Italian series. It is excluded from the primary renewablegeneration aggregate.  
Consequently, the resulting measure is defined as RES_core, representing the selected renewable technologies with known definitions, rather than a complete measure of all renewable generation. This limitation is documented when interpreting the resulting renewablegeneration share.  
5.	**Most of sources lack completeness of data after excluding 2015**  
After excluding 2015, some ENTSOE generation series remain incomplete. The missing observations are not replaced automatically with zero. Instead, the treatment depends on the variable and the analytical purpose.  
For geothermal generation, missing ENTSOE observations within the available TERNA coverage are supplemented with TERNA data after harmonising units and timestamps. The original ENTSOE values are preserved, and the source of each integrated observation is recorded.  
For the remaining technologies, missing values are retained unless their cause can be established or a compatible alternative source is validated. Completeness is assessed again after the documented treatments. Subsequent analyses use observations complete for the variables required by each specification.  


#### A. TERNA geothermal dataset inspection

In [14]:
terna_path = Path('../dataset/raw/terna/generation')
terna = pd.read_excel(terna_path / 'TERNA-generation-test.xlsx', engine='openpyxl')
terna.head()

,datetime_local,Geothermal
0,2024-11-16 00:00:00,0.61
1,2024-11-16 01:00:00,0.62
2,2024-11-16 02:00:00,0.62
3,2024-11-16 03:00:00,0.62
4,2024-11-16 04:00:00,0.62


In [15]:
# parse timestamps and numeric values
terna['datetime_local'] = pd.to_datetime(
    terna['datetime_local'],
    errors='raise'
)

terna['Geothermal'] = pd.to_numeric(
    terna['Geothermal'],
    errors='raise'
)

terna = terna.sort_values('datetime_local')

# basic checks
print('Shape:', terna.shape)
print('Time range:')
print(terna['datetime_local'].min())
print(terna['datetime_local'].max())

print('\nMissing values:')
print(terna.isna().sum())

print('\nDuplicate local timestamps:')
duplicates = terna[
    terna['datetime_local'].duplicated(keep=False)
]
print(duplicates)

print('\nObserved intervals:')
print(
    terna['datetime_local']
    .diff()
    .value_counts()
    .head(10)
)

print('\nGeothermal descriptive statistics:')
print(terna['Geothermal'].describe())

Shape: (53317, 2)
Time range:
2024-11-16 00:00:00
2026-06-30 23:45:00

Missing values:
datetime_local    0
Geothermal        0
dtype: int64

Duplicate local timestamps:
Empty DataFrame
Columns: [datetime_local, Geothermal]
Index: []

Observed intervals:
datetime_local
0 days 00:15:00    52208
0 days 01:00:00     1105
0 days 01:15:00        2
2 days 00:15:00        1
Name: count, dtype: int64

Geothermal descriptive statistics:
count    53317.000000
mean         0.595361
std          0.017061
min          0.440000
25%          0.600000
50%          0.600000
75%          0.600000
max          1.200000
Name: Geothermal, dtype: float64


In [16]:
# sort and reset index
terna = (
    terna.sort_values('datetime_local')
    .reset_index(drop=True)
    .copy()
)

# calculate time intervals
delta = terna['datetime_local'].diff()

# classify each interval
terna['step_type'] = 'other'

terna.loc[
    delta == pd.Timedelta(minutes=15),
    'step_type'
] = '15min'

terna.loc[
    delta == pd.Timedelta(hours=1),
    'step_type'
] = '1h'

terna.loc[
    delta == pd.Timedelta(hours=1, minutes=15),
    'step_type'
] = '1h15'

terna.loc[
    delta == pd.Timedelta(days=2, minutes=15),
    'step_type'
] = '48h15'

# first observation has no preceding interval
terna.loc[delta.isna(), 'step_type'] = 'start'

# create a table of intervals between observations
intervals = pd.DataFrame({
    'start': terna['datetime_local'].shift(),
    'end': terna['datetime_local'],
    'step_type': terna['step_type']
}).iloc[1:].copy()

# identify consecutive blocks with the same frequency
intervals['block'] = (
    intervals['step_type']
    .ne(intervals['step_type'].shift())
    .cumsum()
)

blocks = (
    intervals.groupby('block')
    .agg(
        frequency=('step_type', 'first'),
        start=('start', 'first'),
        end=('end', 'last'),
        n_intervals=('step_type', 'size')
    )
    .reset_index(drop=True)
)

print(blocks.to_string(index=False))

frequency               start                 end  n_intervals
       1h 2024-11-16 00:00:00 2025-01-01 00:00:00         1104
    15min 2025-01-01 00:00:00 2025-03-30 01:45:00         8455
     1h15 2025-03-30 01:45:00 2025-03-30 03:00:00            1
    15min 2025-03-30 03:00:00 2025-10-26 02:00:00        20156
       1h 2025-10-26 02:00:00 2025-10-26 03:00:00            1
    15min 2025-10-26 03:00:00 2026-01-13 23:45:00         7667
    48h15 2026-01-13 23:45:00 2026-01-16 00:00:00            1
    15min 2026-01-16 00:00:00 2026-03-29 01:45:00         6919
     1h15 2026-03-29 01:45:00 2026-03-29 03:00:00            1
    15min 2026-03-29 03:00:00 2026-06-30 23:45:00         9011


In [ ]:
print(
    terna.nlargest(
        10,
        'Geothermal'
    )[['datetime_local', 'Geothermal']]
)

print(
    'Observations above 0.82:',
    (terna['Geothermal'] > 0.82).sum()
)

           datetime_local  Geothermal
29716 2025-10-26 02:00:00        1.20
1386  2025-01-03 22:30:00        0.64
1387  2025-01-03 22:45:00        0.64
1388  2025-01-03 23:00:00        0.64
1389  2025-01-03 23:15:00        0.64
1390  2025-01-03 23:30:00        0.64
1391  2025-01-03 23:45:00        0.64
1392  2025-01-04 00:00:00        0.64
1393  2025-01-04 00:15:00        0.64
1394  2025-01-04 00:30:00        0.64
Observations above 0.82: 1


In [18]:
def inspect_window(df, start, end):
    mask = (
        df['datetime_local'].between(start, end)
    )

    print(
        df.loc[mask, ['datetime_local', 'Geothermal']]
        .to_string(index=False)
    )

# frequency transition
inspect_window(
    terna,
    '2024-12-31 18:00:00',
    '2025-01-01 06:00:00'
)

# Autumn DST transition
inspect_window(
    terna,
    '2025-10-26 00:00:00',
    '2025-10-26 05:00:00'
)

# missing interval in January
inspect_window(
    terna,
    '2026-01-13 18:00:00',
    '2026-01-16 06:00:00'
)

     datetime_local  Geothermal
2024-12-31 18:00:00         0.6
2024-12-31 19:00:00         0.6
2024-12-31 20:00:00         0.6
2024-12-31 21:00:00         0.6
2024-12-31 22:00:00         0.6
2024-12-31 23:00:00         0.6
2025-01-01 00:00:00         0.6
2025-01-01 00:15:00         0.6
2025-01-01 00:30:00         0.6
2025-01-01 00:45:00         0.6
2025-01-01 01:00:00         0.6
2025-01-01 01:15:00         0.6
2025-01-01 01:30:00         0.6
2025-01-01 01:45:00         0.6
2025-01-01 02:00:00         0.6
2025-01-01 02:15:00         0.6
2025-01-01 02:30:00         0.6
2025-01-01 02:45:00         0.6
2025-01-01 03:00:00         0.6
2025-01-01 03:15:00         0.6
2025-01-01 03:30:00         0.6
2025-01-01 03:45:00         0.6
2025-01-01 04:00:00         0.6
2025-01-01 04:15:00         0.6
2025-01-01 04:30:00         0.6
2025-01-01 04:45:00         0.6
2025-01-01 05:00:00         0.6
2025-01-01 05:15:00         0.6
2025-01-01 05:30:00         0.6
2025-01-01 05:45:00         0.6
2025-01-

In [19]:
# 1. inspect the frequency transition
print(
    terna.loc[
        terna['datetime_local'].between(
            '2024-12-31 18:00',
            '2025-01-01 06:00'
        )
    ]
)

# 2. inspect the DST anomaly
print(
    terna.loc[
        terna['datetime_local'].between(
            '2025-10-26 00:00',
            '2025-10-26 05:00'
        )
    ]
)

# 3. inspect the missing dates in January
print(
    terna.loc[
        terna['datetime_local'].between(
            '2026-01-13 18:00',
            '2026-01-16 06:00'
        )
    ]
)

          datetime_local  Geothermal step_type
1098 2024-12-31 18:00:00         0.6        1h
1099 2024-12-31 19:00:00         0.6        1h
1100 2024-12-31 20:00:00         0.6        1h
1101 2024-12-31 21:00:00         0.6        1h
1102 2024-12-31 22:00:00         0.6        1h
1103 2024-12-31 23:00:00         0.6        1h
1104 2025-01-01 00:00:00         0.6        1h
1105 2025-01-01 00:15:00         0.6     15min
1106 2025-01-01 00:30:00         0.6     15min
1107 2025-01-01 00:45:00         0.6     15min
1108 2025-01-01 01:00:00         0.6     15min
1109 2025-01-01 01:15:00         0.6     15min
1110 2025-01-01 01:30:00         0.6     15min
1111 2025-01-01 01:45:00         0.6     15min
1112 2025-01-01 02:00:00         0.6     15min
1113 2025-01-01 02:15:00         0.6     15min
1114 2025-01-01 02:30:00         0.6     15min
1115 2025-01-01 02:45:00         0.6     15min
1116 2025-01-01 03:00:00         0.6     15min
1117 2025-01-01 03:15:00         0.6     15min
1118 2025-01-

In [21]:
# flag the ambiguous DST observation
terna['dst_issue'] = (
    terna['datetime_local']
    == pd.Timestamp('2025-10-26 02:00:00')
)

terna

,datetime_local,Geothermal,step_type,dst_issue
0,2024-11-16 00:00:00,0.61,start,False
1,2024-11-16 01:00:00,0.62,1h,False
2,2024-11-16 02:00:00,0.62,1h,False
3,2024-11-16 03:00:00,0.62,1h,False
4,2024-11-16 04:00:00,0.62,1h,False
...,...,...,...,...
53312,2026-06-30 22:45:00,0.56,15min,False
53313,2026-06-30 23:00:00,0.56,15min,False
53314,2026-06-30 23:15:00,0.56,15min,False
53315,2026-06-30 23:30:00,0.56,15min,False


In [22]:
# convert local timestamps to timezone-aware datetime
terna['datetime_local_tz'] = (
    terna['datetime_local']
    .dt.tz_localize(
        'Europe/Rome',
        ambiguous='NaT',
        nonexistent='NaT'
    )
)

# convert to UTC
terna['datetime_UTC'] = (
    terna['datetime_local_tz']
    .dt.tz_convert('UTC')
)

# inspect timestamps that cannot be assigned unambiguously
print(
    terna.loc[
        terna['datetime_UTC'].isna(),
        ['datetime_local', 'Geothermal']
    ]
)

terna

           datetime_local  Geothermal
29716 2025-10-26 02:00:00         1.2


,datetime_local,Geothermal,step_type,dst_issue,datetime_local_tz,datetime_UTC
0,2024-11-16 00:00:00,0.61,start,False,2024-11-16 00:00:00+01:00,2024-11-15 23:00:00+00:00
1,2024-11-16 01:00:00,0.62,1h,False,2024-11-16 01:00:00+01:00,2024-11-16 00:00:00+00:00
2,2024-11-16 02:00:00,0.62,1h,False,2024-11-16 02:00:00+01:00,2024-11-16 01:00:00+00:00
3,2024-11-16 03:00:00,0.62,1h,False,2024-11-16 03:00:00+01:00,2024-11-16 02:00:00+00:00
4,2024-11-16 04:00:00,0.62,1h,False,2024-11-16 04:00:00+01:00,2024-11-16 03:00:00+00:00
...,...,...,...,...,...,...
53312,2026-06-30 22:45:00,0.56,15min,False,2026-06-30 22:45:00+02:00,2026-06-30 20:45:00+00:00
53313,2026-06-30 23:00:00,0.56,15min,False,2026-06-30 23:00:00+02:00,2026-06-30 21:00:00+00:00
53314,2026-06-30 23:15:00,0.56,15min,False,2026-06-30 23:15:00+02:00,2026-06-30 21:15:00+00:00
53315,2026-06-30 23:30:00,0.56,15min,False,2026-06-30 23:30:00+02:00,2026-06-30 21:30:00+00:00


#### B. Final merge with TERNA data

In [23]:
# prepare TERNA geothermal data
terna_clean = terna[
    ['datetime_local', 'Geothermal']
].copy()

terna_clean['datetime_local'] = pd.to_datetime(
    terna_clean['datetime_local'],
    errors='raise'
)

terna_clean['Geothermal'] = pd.to_numeric(
    terna_clean['Geothermal'],
    errors='raise'
)

# convert local timestamps to timezone-aware timestamps
terna_clean['datetime_local_tz'] = (
    terna_clean['datetime_local'].dt.tz_localize(
        'Europe/Rome',
        ambiguous='NaT',
        nonexistent='NaT'
    )
)

# exclude timestamps that cannot be localized unambiguously
n_before = len(terna_clean)

terna_clean = terna_clean.dropna(
    subset=['datetime_local_tz']
).copy()

print(
    'Rows excluded due to ambiguous/nonexistent local time:',
    n_before - len(terna_clean)
)

# convert timestamps to UTC
terna_clean['datetime_UTC'] = (
    terna_clean['datetime_local_tz'].dt.tz_convert('UTC')
)

# convert GW to MW
terna_clean['Geothermal_TERNA_MW'] = (
    terna_clean['Geothermal'] * 1000
)

Rows excluded due to ambiguous/nonexistent local time: 1


In [24]:
cutoff = pd.Timestamp('2025-01-01 00:00:00')

# original hourly observations
hourly_part = terna_clean.loc[
    terna_clean['datetime_local'] < cutoff,
    ['datetime_UTC', 'Geothermal_TERNA_MW']
].copy()

# original quarter-hourly observations
quarter_part = terna_clean.loc[
    terna_clean['datetime_local'] >= cutoff
].copy()

quarter_part = quarter_part.set_index('datetime_UTC')

In [25]:
# aggregate quarter-hourly power to hourly mean power

# keep an hour only when all four quarter-hour observations exist
quarter_stats = (
    quarter_part['Geothermal_TERNA_MW']
    .resample('h')
    .agg(['mean', 'count'])
)

quarter_hourly = (
    quarter_stats.loc[
        quarter_stats['count'] == 4,
        ['mean']
    ]
    .rename(columns={'mean': 'Geothermal_TERNA_MW'})
    .reset_index()
)

In [26]:
# combine the original hourly block and the aggregated block
terna_hourly = pd.concat(
    [hourly_part, quarter_hourly],
    ignore_index=True
).sort_values('datetime_UTC').reset_index(drop=True)

print('TERNA hourly dataset:', terna_hourly.shape)
print(
    terna_hourly['datetime_UTC'].min(),
    '→',
    terna_hourly['datetime_UTC'].max()
)

print('Duplicated UTC timestamps:',
      terna_hourly['datetime_UTC'].duplicated().sum())

TERNA hourly dataset: (14157, 2)
2024-11-15 23:00:00+00:00 → 2026-06-30 21:00:00+00:00
Duplicated UTC timestamps: 0


In [30]:
# execute the final merge

entsoe = df_test.copy()

# preserve the original ENTSO-E values
entsoe['Geothermal_integrated'] = entsoe['Geothermal']
entsoe['Geothermal_source'] = 'ENTSO-E'

# merge the TERNA hourly series
entsoe = entsoe.merge(
    terna_hourly,
    on='datetime_UTC',
    how='left',
    validate='one_to_one'
)

# fill only where ENTSO-E is missing and TERNA is available
fill_mask = (
    entsoe['Geothermal_integrated'].isna()
    & entsoe['Geothermal_TERNA_MW'].notna()
)

entsoe.loc[
    fill_mask,
    'Geothermal_integrated'
] = entsoe.loc[
    fill_mask,
    'Geothermal_TERNA_MW'
]

entsoe.loc[
    fill_mask,
    'Geothermal_source'
] = 'TERNA'

# label observations that remain missing
entsoe.loc[
    entsoe['Geothermal_integrated'].isna(),
    'Geothermal_source'
] = 'MISSING'

In [33]:
# verify the results
print('Original ENTSO-E missing:', entsoe['Geothermal'].isna().sum())
print('Observations filled by TERNA:', fill_mask.sum())
print('Missing after integration:', entsoe['Geothermal_integrated'].isna().sum())
print('\nSource distribution:')
print(entsoe['Geothermal_source'].value_counts())

assert entsoe.loc[
    entsoe['Geothermal_source'] == 'TERNA',
    'Geothermal'
].isna().all()

Original ENTSO-E missing: 8147
Observations filled by TERNA: 8099
Missing after integration: 48

Source distribution:
Geothermal_source
ENTSO-E    83868
TERNA       8099
MISSING       48
Name: count, dtype: int64


In [38]:
# get copmleteness rate by year from generation columns
entsoe['year'] = entsoe['datetime_UTC'].dt.year

new_completeness = pd.DataFrame(
    index=sorted(entsoe['year'].unique())
)

for col in generation_cols:
    new_completeness[col] = (
        entsoe.groupby('year')[col]
        .apply(lambda x: round((x.notna().mean() * 100), 2))
    )
new_completeness = new_completeness.reset_index()
new_completeness = new_completeness.rename(columns={'index': 'year'})

new_completeness

,year,Biomass,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,Geothermal,Hydro Pumped Storage,Hydro Run-of-river and poundage,Hydro Water Reservoir,Other,Solar,Waste,Wind Onshore,Wind Offshore,Energy storage,Other renewable
0,2015,100.0,100.00,100.00,100.00,100.00,100.00,100.00,100.0,100.00,100.00,100.0,100.00,100.00,100.00,0.00,0.00
1,2016,100.0,100.00,98.62,100.00,100.00,100.00,99.76,100.0,100.00,100.00,100.0,100.00,100.00,100.00,0.00,0.00
2,2017,100.0,100.00,99.73,98.90,100.00,100.00,99.89,100.0,100.00,100.00,100.0,100.00,100.00,100.00,0.00,0.00
3,2018,100.0,99.73,100.00,99.73,100.00,100.00,99.99,100.0,100.00,98.08,100.0,100.00,100.00,100.00,0.00,0.00
4,2019,100.0,93.70,100.00,100.00,99.73,100.00,97.15,100.0,100.00,98.63,100.0,100.00,100.00,100.00,0.00,0.00
5,2020,100.0,100.00,100.00,100.00,100.00,100.00,95.34,100.0,100.00,100.00,100.0,100.00,100.00,100.00,0.00,0.00
6,2021,100.0,100.00,100.00,100.00,100.00,100.00,95.51,100.0,99.73,100.00,100.0,100.00,100.00,100.00,0.00,0.00
7,2022,100.0,100.00,100.00,100.00,100.00,100.00,93.79,100.0,100.00,100.00,100.0,100.00,100.00,90.14,0.00,0.00
8,2023,100.0,100.00,100.00,100.00,100.00,100.00,94.98,100.0,100.00,100.00,100.0,100.00,100.00,100.00,0.00,0.00
9,2024,100.0,100.00,100.00,100.00,91.79,91.25,95.09,100.0,100.00,100.00,100.0,100.00,100.00,100.00,0.01,0.01


In [36]:
entsoe

,datetime_UTC,datetime_local,local_hour,PUN,National Load,Biomass,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,...,Solar,Waste,Wind Onshore,Wind Offshore,Energy storage,Other renewable,year,Geothermal_integrated,Geothermal_source,Geothermal_TERNA_MW
0,2015-12-31 23:00:00+00:00,2016-01-01 00:00:00+01:00,0,50.41211,23927.233,220.00,241.00,5588.00,1686.00,542.00,...,0.00,45.0,203.00,0.00,NaN,NaN,2015,706.00,ENTSO-E,NaN
1,2016-01-01 00:00:00+00:00,2016-01-01 01:00:00+01:00,1,44.76435,22517.279,216.00,241.00,5034.00,1719.00,519.00,...,1.00,46.0,218.00,0.00,NaN,NaN,2016,706.00,ENTSO-E,NaN
2,2016-01-01 01:00:00+00:00,2016-01-01 02:00:00+01:00,2,40.01613,21354.678,217.00,241.00,4486.00,1728.00,530.00,...,0.00,44.0,267.00,0.00,NaN,NaN,2016,706.00,ENTSO-E,NaN
3,2016-01-01 02:00:00+00:00,2016-01-01 03:00:00+01:00,3,37.92076,20219.874,216.00,241.00,4275.00,1727.00,467.00,...,0.00,39.0,227.00,0.00,NaN,NaN,2016,705.00,ENTSO-E,NaN
4,2016-01-01 03:00:00+00:00,2016-01-01 04:00:00+01:00,4,35.77287,19229.221,215.00,241.00,3926.00,1658.00,426.00,...,0.00,44.0,212.00,0.00,NaN,NaN,2016,707.00,ENTSO-E,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92010,2026-06-30 17:00:00+00:00,2026-06-30 19:00:00+02:00,19,204.10622,48437.300,712.50,641.50,26968.25,394.75,155.75,...,1755.75,NaN,1780.75,1.00,1148.333333,NaN,2026,553.00,ENTSO-E,560.0
92011,2026-06-30 18:00:00+00:00,2026-06-30 20:00:00+02:00,20,198.16000,47383.500,714.00,661.75,27442.75,396.50,156.00,...,252.00,NaN,2076.50,5.75,1273.750000,NaN,2026,552.25,ENTSO-E,560.0
92012,2026-06-30 19:00:00+00:00,2026-06-30 21:00:00+02:00,21,182.03250,45951.100,716.50,676.50,27084.00,392.25,156.00,...,0.50,NaN,2587.75,5.50,830.333333,NaN,2026,554.25,ENTSO-E,560.0
92013,2026-06-30 20:00:00+00:00,2026-06-30 22:00:00+02:00,22,173.30306,43204.200,710.25,647.50,26959.75,365.75,158.25,...,0.00,NaN,2068.75,1.25,241.750000,NaN,2026,555.00,ENTSO-E,560.0


### 1.4. Post-treatment validation

In [39]:
# create validation copy
post = entsoe.copy()

# use integrated geothermal series as the canonical variable
post['Geothermal'] = post['Geothermal_integrated']

# Define the year according to the Italian local calendar
post['year'] = post['datetime_local'].dt.year

post

,datetime_UTC,datetime_local,local_hour,PUN,National Load,Biomass,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,...,Solar,Waste,Wind Onshore,Wind Offshore,Energy storage,Other renewable,year,Geothermal_integrated,Geothermal_source,Geothermal_TERNA_MW
0,2015-12-31 23:00:00+00:00,2016-01-01 00:00:00+01:00,0,50.41211,23927.233,220.00,241.00,5588.00,1686.00,542.00,...,0.00,45.0,203.00,0.00,NaN,NaN,2016,706.00,ENTSO-E,NaN
1,2016-01-01 00:00:00+00:00,2016-01-01 01:00:00+01:00,1,44.76435,22517.279,216.00,241.00,5034.00,1719.00,519.00,...,1.00,46.0,218.00,0.00,NaN,NaN,2016,706.00,ENTSO-E,NaN
2,2016-01-01 01:00:00+00:00,2016-01-01 02:00:00+01:00,2,40.01613,21354.678,217.00,241.00,4486.00,1728.00,530.00,...,0.00,44.0,267.00,0.00,NaN,NaN,2016,706.00,ENTSO-E,NaN
3,2016-01-01 02:00:00+00:00,2016-01-01 03:00:00+01:00,3,37.92076,20219.874,216.00,241.00,4275.00,1727.00,467.00,...,0.00,39.0,227.00,0.00,NaN,NaN,2016,705.00,ENTSO-E,NaN
4,2016-01-01 03:00:00+00:00,2016-01-01 04:00:00+01:00,4,35.77287,19229.221,215.00,241.00,3926.00,1658.00,426.00,...,0.00,44.0,212.00,0.00,NaN,NaN,2016,707.00,ENTSO-E,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92010,2026-06-30 17:00:00+00:00,2026-06-30 19:00:00+02:00,19,204.10622,48437.300,712.50,641.50,26968.25,394.75,155.75,...,1755.75,NaN,1780.75,1.00,1148.333333,NaN,2026,553.00,ENTSO-E,560.0
92011,2026-06-30 18:00:00+00:00,2026-06-30 20:00:00+02:00,20,198.16000,47383.500,714.00,661.75,27442.75,396.50,156.00,...,252.00,NaN,2076.50,5.75,1273.750000,NaN,2026,552.25,ENTSO-E,560.0
92012,2026-06-30 19:00:00+00:00,2026-06-30 21:00:00+02:00,21,182.03250,45951.100,716.50,676.50,27084.00,392.25,156.00,...,0.50,NaN,2587.75,5.50,830.333333,NaN,2026,554.25,ENTSO-E,560.0
92013,2026-06-30 20:00:00+00:00,2026-06-30 22:00:00+02:00,22,173.30306,43204.200,710.25,647.50,26959.75,365.75,158.25,...,0.00,NaN,2068.75,1.25,241.750000,NaN,2026,555.00,ENTSO-E,560.0


In [ ]:
# 1. TEMPORAL INTEGRITY
# Define the analysis sample using Italian local time
start_local = pd.Timestamp(
    '2016-01-01 00:00:00',
    tz='Europe/Rome'
)

end_local = pd.Timestamp(
    '2026-07-01 00:00:00',
    tz='Europe/Rome'
)

# convert sample boundaries to UTC
start_utc = start_local.tz_convert('UTC')
end_utc_exclusive = end_local.tz_convert('UTC')

# expected hourly timestamps
expected_utc = pd.date_range(
    start=start_utc,
    end=end_utc_exclusive - pd.Timedelta(hours=1),
    freq='h'
)

actual_utc = pd.DatetimeIndex(post['datetime_UTC'])

print('Observations:', len(post))
print('Expected observations:', len(expected_utc))
print('First UTC timestamp:', post['datetime_UTC'].min())
print('Last UTC timestamp:', post['datetime_UTC'].max())

# check timestamps
assert post['datetime_UTC'].notna().all(), \
    'Missing UTC timestamps found'

assert not post['datetime_UTC'].duplicated().any(), \
    'Duplicate UTC timestamps found'

assert post['datetime_UTC'].is_monotonic_increasing, \
    'UTC timestamps are not sorted'

assert actual_utc.equals(expected_utc), \
    'The dataset does not contain the complete expected hourly index'

print('MSG: temporal integrity validation passed')

Observations: 92015
Expected observations: 92015
First UTC timestamp: 2015-12-31 23:00:00+00:00
Last UTC timestamp: 2026-06-30 21:00:00+00:00
MSG: temporal integrity validation passed


In [41]:
# 2. UTC AND LOCAL CONSISTENCY
# Reconstruct local timestamps from UTC
expected_local = (
    post['datetime_UTC']
    .dt.tz_convert('Europe/Rome')
)

assert (
    post['datetime_local'].reset_index(drop=True)
    == expected_local.reset_index(drop=True)
).all(), 'Local timestamps are inconsistent with UTC'

assert (
    post['local_hour']
    == post['datetime_local'].dt.hour
).all(), 'local_hour is inconsistent with datetime_local'

assert (
    (post['datetime_local'] >= start_local)
    & (post['datetime_local'] < end_local)
).all(), 'Observations outside the analysis sample found'

print('MSG: timezone and sample-boundary validation passed')

MSG: timezone and sample-boundary validation passed


In [42]:
# 3. MISSING DATA TRANSOFRMATION VALIDATION

## Wind Offshore
cutoff = pd.Timestamp(
    '2022-04-21 00:00:00',
    tz='Europe/Rome'
)

pre_cutoff = post['datetime_local'] < cutoff

print(
    'Wind Offshore NaN before cutoff:',
    post.loc[pre_cutoff, 'Wind Offshore'].isna().sum()
)

print(
    'Wind Offshore NaN from cutoff onward:',
    post.loc[~pre_cutoff, 'Wind Offshore'].isna().sum()
)

assert post.loc[
    pre_cutoff, 'Wind Offshore'
].notna().all(), \
    'Missing Wind Offshore values remain before the cutoff'

print('MSG: Wind Offshore treatment validation passed')

Wind Offshore NaN before cutoff: 0
Wind Offshore NaN from cutoff onward: 864
MSG: Wind Offshore treatment validation passed


In [43]:
## Geothermal
source_entsoe = entsoe['Geothermal_source'].eq('ENTSO-E')
source_terna = entsoe['Geothermal_source'].eq('TERNA')
source_missing = entsoe['Geothermal_source'].eq('MISSING')

# Original ENTSO-E values must remain unchanged
assert (
    entsoe.loc[source_entsoe, 'Geothermal'].to_numpy()
    == entsoe.loc[source_entsoe, 'Geothermal_integrated'].to_numpy()
).all(), 'Original ENTSO-E geothermal values were changed'

# TERNA must fill only observations missing in ENTSO-E
assert entsoe.loc[
    source_terna, 'Geothermal'
].isna().all(), \
    'TERNA was used where ENTSO-E already had a value'

assert (
    entsoe.loc[source_terna, 'Geothermal_integrated'].to_numpy()
    == entsoe.loc[source_terna, 'Geothermal_TERNA_MW'].to_numpy()
).all(), 'Integrated TERNA values do not match their source'

# Remaining missing values must be labelled accordingly
assert entsoe.loc[
    source_missing, 'Geothermal_integrated'
].isna().all(), \
    'Missing geothermal values are not correctly labelled'

print('MSG: geothermal integration validation passed')

MSG: geothermal integration validation passed


In [44]:
# 4. COMPLETENESS AFTER TREATMENS
# baseline: original dataset before missing-data treatments
before = df.copy()
before['year'] = before['datetime_local'].dt.year

before_completeness = (
    before.groupby('year')[generation_cols]
    .agg(lambda x: x.notna().mean() * 100)
    .round(2)
)

# after: harmonized series and documented treatments
after_completeness = (
    post.groupby('year')[generation_cols]
    .agg(lambda x: x.notna().mean() * 100)
    .round(2)
)

# improvement in percentage points
completeness_change = (
    after_completeness - before_completeness
).round(2)

print('--- COMPLETENESS AFTER TREATMENT (%) ---')
display(after_completeness)

print('--- CHANGE IN COMPLETENESS (percentage points) ---')
display(completeness_change)

--- COMPLETENESS AFTER TREATMENT (%) ---


,Biomass,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,Geothermal,Hydro Pumped Storage,Hydro Run-of-river and poundage,Hydro Water Reservoir,Other,Solar,Waste,Wind Onshore,Wind Offshore,Energy storage,Other renewable
year,,,,,,,,,,,,,,,,
2016,100.0,100.00,98.62,100.00,100.00,100.00,99.76,100.0,100.00,100.00,100.0,100.00,100.00,100.00,0.00,0.00
2017,100.0,100.00,99.73,98.90,100.00,100.00,99.89,100.0,100.00,100.00,100.0,100.00,100.00,100.00,0.00,0.00
2018,100.0,99.73,100.00,99.73,100.00,100.00,99.99,100.0,100.00,98.08,100.0,100.00,100.00,100.00,0.00,0.00
2019,100.0,93.70,100.00,100.00,99.73,100.00,97.15,100.0,100.00,98.63,100.0,100.00,100.00,100.00,0.00,0.00
2020,100.0,100.00,100.00,100.00,100.00,100.00,95.34,100.0,100.00,100.00,100.0,100.00,100.00,100.00,0.00,0.00
2021,100.0,100.00,100.00,100.00,100.00,100.00,95.51,100.0,99.73,100.00,100.0,100.00,100.00,100.00,0.00,0.00
2022,100.0,100.00,100.00,100.00,100.00,100.00,93.79,100.0,100.00,100.00,100.0,100.00,100.00,90.14,0.00,0.00
2023,100.0,100.00,100.00,100.00,100.00,100.00,94.98,100.0,100.00,100.00,100.0,100.00,100.00,100.00,0.00,0.00
2024,100.0,100.00,100.00,100.00,91.79,100.00,95.09,100.0,100.00,100.00,100.0,100.00,100.00,100.00,0.00,0.00


--- CHANGE IN COMPLETENESS (percentage points) ---


,Biomass,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,Geothermal,Hydro Pumped Storage,Hydro Run-of-river and poundage,Hydro Water Reservoir,Other,Solar,Waste,Wind Onshore,Wind Offshore,Energy storage,Other renewable
year,,,,,,,,,,,,,,,,
2016,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0,0.0
2017,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0,0.0
2018,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0,0.0
2019,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0,0.0
2020,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0,0.0
2021,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.00,0.0,0.0
2022,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.13,0.0,0.0
2023,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0
2024,0.0,0.0,0.0,0.0,0.0,8.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0


### 1.5. Export analysis ready dataset

In [62]:
post.columns

Index(['datetime_UTC', 'datetime_local', 'local_hour', 'PUN', 'National Load',
       'Biomass', 'Fossil Coal-derived gas', 'Fossil Gas', 'Fossil Hard coal',
       'Fossil Oil', 'Geothermal', 'Hydro Pumped Storage',
       'Hydro Run-of-river and poundage', 'Hydro Water Reservoir', 'Other',
       'Solar', 'Waste', 'Wind Onshore', 'Wind Offshore', 'Energy storage',
       'Other renewable', 'Geothermal_integrated', 'Geothermal_source',
       'Geothermal_TERNA_MW'],
      dtype='str')

In [67]:
# remove intermediate columns
post_final = post.drop(
    columns=[
        'year',
        'Geothermal_integrated',
        'Geothermal_TERNA_MW'
    ],
    errors='ignore'
).copy()

In [70]:
output_path = Path('../dataset/filled')

# create folder if non-existent
output_path.mkdir(parents=True, exist_ok=True)

# save in CSV format
post_final.to_csv(output_path / 'IT.csv', index=False)